# Apple Plant Disease XAI — Four-Sample HITL Demonstration

This notebook creates four presentation-ready reports using the existing Notebook 4 trust report and Notebook 6 explanation outputs: **three low-trust samples requiring human review** and **one high-trust sample requiring no review**. Each report displays the original image and a saved Grad-CAM overlay side by side, prediction/trust metrics, the saved explanation, and review status.

For the three low-trust samples, it prints a human-review prompt, accepts actual typed feedback (`accept`, `correct`, or `reject`), echoes the response, and persists it. The high-trust case is clearly marked **HUMAN REVIEW: NOT REQUIRED**.

**Important:** image indices must map to the exact Notebook 4 test-set ordering. This notebook refuses to guess or substitute demo images. Configure the manifest/overrides below if the original project has not saved image paths.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
import torch
import torch.nn as nn
from torchvision import models
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/XAI_Project")
DATA_DIR = BASE_DIR / "processed"
MODEL_DIR = BASE_DIR / "saved_models"
RESULTS_DIR = BASE_DIR / "results"
TRUST_REPORT_PATH = RESULTS_DIR / "trust_uncertainty" / "trust_uncertainty_test_report.csv"
LLM_DIR = RESULTS_DIR / "llm_explanations"
HF_DIR = RESULTS_DIR / "human_feedback"
HF_DIR.mkdir(parents=True, exist_ok=True)

# Notebook 4 evaluates processed/test.pt with shuffle=False. Therefore report row
# image_index i maps directly to test_images[i]; no image-path manifest is needed.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4
IMAGE_SIZE = 224
CLASS_NAMES = {0:"Apple Healthy", 1:"Apple Scab", 2:"Apple Black Rot", 3:"Apple Cedar Apple Rust"}

TRUST_REVIEW_THRESHOLD = 0.70
CONFIDENCE_REVIEW_THRESHOLD = 0.80
ENTROPY_REVIEW_THRESHOLD = 0.50
MARGIN_REVIEW_THRESHOLD = 0.20
INTERACTIVE_FEEDBACK = True

print("Project:", BASE_DIR, "| exists:", BASE_DIR.exists())
print("Device:", DEVICE)


## 1. Load the existing trust report and Notebook 6 explanations

In [ ]:
if not TRUST_REPORT_PATH.exists():
    raise FileNotFoundError(f"Trust report not found: {TRUST_REPORT_PATH}. Run Notebook 4 first.")
trust_df = pd.read_csv(TRUST_REPORT_PATH)
required = ["image_index","true_class","predicted_class","confidence","entropy","margin","trust_score","trust_level","predicted_label"]
missing = [c for c in required if c not in trust_df.columns]
if missing: raise ValueError(f"Trust report missing columns: {missing}")
trust_df["image_index"] = pd.to_numeric(trust_df["image_index"], errors="raise").astype(int)
for c in ["confidence","entropy","margin","trust_score"]: trust_df[c] = pd.to_numeric(trust_df[c], errors="coerce")
if trust_df["image_index"].duplicated().any(): raise ValueError("Duplicate image_index values in trust report.")
def review_reasons(r):
    out=[]
    if float(r.trust_score) < TRUST_REVIEW_THRESHOLD: out.append("low_trust")
    if float(r.confidence) < CONFIDENCE_REVIEW_THRESHOLD: out.append("low_confidence")
    if float(r.entropy) > ENTROPY_REVIEW_THRESHOLD: out.append("high_entropy")
    if float(r.margin) < MARGIN_REVIEW_THRESHOLD: out.append("low_margin")
    return out
trust_df["review_reasons"] = trust_df.apply(review_reasons, axis=1)
trust_df["requires_human_review"] = trust_df["review_reasons"].apply(bool)

llm_candidates=[LLM_DIR/"llm_explanations_all_test_samples.csv", LLM_DIR/"llm_30_samples"/"llm_explanations_30_samples.csv"]
llm_path=next((p for p in llm_candidates if p.exists()),None)
llm_df=pd.DataFrame()
if llm_path:
    llm_df=pd.read_csv(llm_path)
    if "image_index" not in llm_df.columns and "sample_id" in llm_df.columns:
        llm_df["image_index"]=pd.to_numeric(llm_df["sample_id"].astype(str).str.extract(r"(\d+)",expand=False),errors="coerce")
    if "image_index" in llm_df.columns:
        llm_df=llm_df[pd.to_numeric(llm_df["image_index"],errors="coerce").notna()].copy()
        llm_df["image_index"]=pd.to_numeric(llm_df["image_index"]).astype(int)
        llm_df=llm_df.drop_duplicates("image_index",keep="last")
    print("Loaded LLM output:",llm_path,"rows:",len(llm_df))
else: print("No saved Notebook 6 explanation CSV found; missing explanations will be reported honestly.")
low=trust_df[trust_df.requires_human_review].sort_values(["trust_score","confidence"],ascending=True)
high=trust_df[~trust_df.requires_human_review].sort_values(["trust_score","confidence"],ascending=False)
if len(low)<3: raise ValueError(f"Only {len(low)} samples meet review criteria; need three.")
if high.empty: raise ValueError("No high-trust sample qualifies as no-review.")
selected=pd.concat([low.head(3),high.head(1)],ignore_index=True)
selected["demo_case"]=["LOW-TRUST SAMPLE 1","LOW-TRUST SAMPLE 2","LOW-TRUST SAMPLE 3","HIGH-TRUST SAMPLE"]
display(selected[["demo_case","image_index","predicted_class","confidence","entropy","margin","trust_score","trust_level","requires_human_review","review_reasons"]])


## 2. Resolve exact test image paths safely

In [ ]:
# Reconstruct the exact test-index mapping used by Notebook 4.
def load_processed_split(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Processed split not found: {path}. Run Notebook 4 data preparation first.")
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(path, map_location="cpu")
    if isinstance(obj, (tuple, list)) and len(obj) == 2:
        images, labels = obj
    elif isinstance(obj, dict):
        x_key = next((k for k in ["images", "X", "data", "inputs"] if k in obj), None)
        y_key = next((k for k in ["labels", "y", "targets"] if k in obj), None)
        if x_key is None or y_key is None:
            raise ValueError(f"Could not identify image/label keys in {path}; keys={list(obj)}")
        images, labels = obj[x_key], obj[y_key]
    else:
        raise ValueError(f"Unsupported processed split format: {type(obj)}")
    return torch.as_tensor(images).float(), torch.as_tensor(labels).long()

test_images, test_labels = load_processed_split("test.pt")
if test_images.ndim != 4:
    raise ValueError(f"Expected test tensor shape [N,C,H,W], got {tuple(test_images.shape)}")
if test_images.shape[1] != 3:
    raise ValueError(f"Expected 3-channel test tensor, got shape {tuple(test_images.shape)}")

# Reconstruct the same four-class ResNet50 architecture and checkpoint candidates as Notebook 4.
def find_checkpoint():
    candidates = [MODEL_DIR / "best_overall_model.pt", MODEL_DIR / "resnet50_best.pt", MODEL_DIR / "best_resnet50.pt"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No ResNet50 checkpoint found. Checked: " + ", ".join(map(str, candidates)))

checkpoint_path = find_checkpoint()
try:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint: state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint: state_dict = checkpoint["state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict): state_dict = checkpoint["model"]
    else: state_dict = checkpoint
else:
    state_dict = checkpoint
state_dict = {(k.replace("module.", "", 1) if k.startswith("module.") else k): v for k, v in state_dict.items()}
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys or unexpected_keys:
    print("Checkpoint key warning | missing:", missing_keys, "| unexpected:", unexpected_keys)
model = model.to(DEVICE).eval()
print("Loaded test tensor:", tuple(test_images.shape), "| labels:", tuple(test_labels.shape))
print("Loaded ResNet50 checkpoint:", checkpoint_path)

# Validate report indices against the exact processed test tensor order.
invalid = [int(i) for i in selected.image_index if int(i) < 0 or int(i) >= len(test_images)]
if invalid:
    raise IndexError(f"Trust report contains indices outside processed/test.pt: {invalid}; test rows={len(test_images)}")
selected["image_ref"] = selected.image_index.map(lambda i: f"processed/test.pt[index={int(i)}]")
display(selected[["demo_case", "image_index", "image_ref"]])


## 3. Render the four screenshot-style reports

In [ ]:
# Notebook 4 stores normalized tensors (ImageNet mean/std). Convert a copy back to RGB for display.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def tensor_to_pil(tensor):
    x = tensor.detach().cpu().float()
    # Inverse ImageNet normalization for visualization; does not change model input.
    x = (x * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)
    arr = (x.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    return Image.fromarray(arr)

def make_gradcam(tensor, class_idx=None):
    """Generate Grad-CAM for the supplied normalized test tensor and loaded ResNet50."""
    activations, gradients = {}, {}
    target_layer = model.layer4[-1]
    def forward_hook(module, inputs, output): activations["value"] = output
    def backward_hook(module, grad_input, grad_output): gradients["value"] = grad_output[0]
    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)
    try:
        model.zero_grad(set_to_none=True)
        x = tensor.unsqueeze(0).to(DEVICE).float()
        logits = model(x)
        if class_idx is None: class_idx = int(logits.argmax(dim=1).item())
        logits[0, class_idx].backward()
        acts = activations["value"].detach()
        grads = gradients["value"].detach()
        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=tensor.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.detach().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

def get_explanation(idx):
    if llm_df.empty or "image_index" not in llm_df.columns:
        return "No saved Notebook 6 explanation available for this sample. No LLM response was fabricated."
    rr = llm_df[llm_df.image_index == int(idx)]
    if rr.empty: return "No saved Notebook 6 explanation available for this sample."
    r = rr.iloc[-1]
    fields = ["summary", "prediction_explanation", "visual_evidence", "knowledge_evidence", "uncertainty_note", "human_review_recommendation"]
    parts = [f"{c.replace('_',' ').title()}: {r[c]}" for c in fields if c in r.index and pd.notna(r[c]) and str(r[c]).strip()]
    if not parts:
        for c in ["llm_explanation_json", "explanation", "generated_explanation"]:
            if c in r.index and pd.notna(r[c]) and str(r[c]).strip(): parts = [str(r[c])]; break
    return "\n\n".join(parts) if parts else "Saved LLM row has no recognized explanation fields."

def show_case(r):
    idx = int(r.image_index)
    x = test_images[idx]
    original = tensor_to_pil(x)
    heat = make_gradcam(x, int(r.predicted_label))
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(original)
    ax[0].set_title(f"Input Image\nPrediction: {r.predicted_class} | Confidence: {float(r.confidence):.2%}")
    ax[0].axis("off")
    ax[1].imshow(original)
    ax[1].imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    ax[1].set_title("Grad-CAM — Influential Regions")
    ax[1].axis("off")
    plt.tight_layout(); plt.show()

    print("RESNET50 + TRUST")
    print(f"Case: {r.demo_case} | Sample: test_{idx:05d} | Image index: {idx}")
    print(f"Prediction: {r.predicted_class} | Actual class: {r.true_class}")
    print(f"Confidence: {float(r.confidence):.2%} | Entropy: {float(r.entropy):.4f} | Margin: {float(r.margin):.4f}")
    print(f"Trust: {float(r.trust_score):.4f} ({r.trust_level})")
    print("Human review:", "REQUIRED" if r.requires_human_review else "NOT REQUIRED")
    print("\nQWEN GROUNDED EXPLANATION (saved Notebook 6 output)")
    print(get_explanation(idx))
for _, r in selected.iterrows():
    display(Markdown("## " + r.demo_case))
    show_case(r)


## 4. Human feedback — actual typed responses

In [ ]:
VALID_DECISIONS={"accept","correct","reject"}
FEEDBACK_JSONL=HF_DIR/"feedback_records.jsonl"; FEEDBACK_CSV=HF_DIR/"feedback_records.csv"
def now_iso(): return datetime.now(timezone.utc).isoformat()
def collect_feedback(r):
    idx=int(r.image_index); pred=str(r.predicted_class)
    print("\n"+"="*70+"\nHUMAN REVIEW REQUIRED")
    print(f"Sample ID: test_{idx:05d}\nPredicted disease: {pred}\nConfidence: {float(r.confidence):.2%}\nTrust score: {float(r.trust_score):.4f} ({r.trust_level})")
    print("Please inspect the image, Grad-CAM, and explanation. Choose ACCEPT, CORRECT, or REJECT.")
    if not INTERACTIVE_FEEDBACK: raise RuntimeError("Interactive feedback is disabled; this notebook will not fabricate a response.")
    while True:
        raw=input("Enter decision (accept/correct/reject): ").strip(); decision=raw.lower()
        if decision in VALID_DECISIONS: break
        print("Invalid choice. Type accept, correct, or reject.")
    corrected=None
    if decision=="correct":
        print("Valid labels:"," | ".join(CLASS_NAMES.values()))
        while True:
            typed_label=input("Enter corrected disease label exactly as shown: ").strip()
            if typed_label in CLASS_NAMES.values(): corrected=typed_label; break
            print("Label not recognized; choose one of the displayed labels.")
    comment=input("Optional reviewer comment (press Enter to skip): ").strip()
    rec={"sample_id":f"test_{idx:05d}","image_index":idx,"image_path":f"processed/test.pt[index={idx}]","model_prediction_class":pred,"true_class":str(r.true_class),"model_confidence":float(r.confidence),"entropy":float(r.entropy),"margin":float(r.margin),"trust_score":float(r.trust_score),"trust_level":str(r.trust_level),"review_reasons":list(r.review_reasons),"human_decision":decision,"corrected_class":corrected if decision=="correct" else (pred if decision=="accept" else None),"human_comment":comment or None,"feedback_source":"user","review_timestamp":now_iso(),"feedback_version":1}
    print("\nHUMAN FEEDBACK — RESPONSE CAPTURED"); print("Exact typed decision:",raw)
    if decision=="correct": print("Exact typed corrected label:",typed_label)
    print("Stored decision:",rec["human_decision"]); print("Stored corrected label:",rec["corrected_class"] or "N/A"); print("Stored comment:",rec["human_comment"] or "N/A")
    return rec
new_feedback=[]
for _,r in selected[selected.requires_human_review].iterrows(): new_feedback.append(collect_feedback(r))
if new_feedback:
    with open(FEEDBACK_JSONL,"a",encoding="utf-8") as f:
        for rec in new_feedback: f.write(json.dumps(rec,ensure_ascii=False)+"\n")
    old=pd.read_csv(FEEDBACK_CSV) if FEEDBACK_CSV.exists() else pd.DataFrame()
    pd.concat([old,pd.DataFrame(new_feedback)],ignore_index=True).to_csv(FEEDBACK_CSV,index=False,encoding="utf-8")
print("\nFeedback records added:",len(new_feedback)); print("JSONL:",FEEDBACK_JSONL); print("CSV:",FEEDBACK_CSV)


## 5. High-trust status and final summary

In [ ]:
high=selected[~selected.requires_human_review].iloc[0]
print("="*70+"\nHIGH-TRUST SAMPLE")
print(f"Sample ID: test_{int(high.image_index):05d} | Prediction: {high.predicted_class}")
print(f"Confidence: {float(high.confidence):.2%} | Trust: {float(high.trust_score):.4f} ({high.trust_level})")
print("HUMAN REVIEW: NOT REQUIRED")
print("No mandatory feedback was requested and no human response was fabricated.")
fb={x["sample_id"]:x for x in new_feedback}; rows=[]
for _,r in selected.iterrows():
    sid=f"test_{int(r.image_index):05d}"; f=fb.get(sid); exp=get_explanation(int(r.image_index))
    rows.append({"case":r.demo_case,"sample_id":sid,"image_index":int(r.image_index),"actual_class":r.true_class,"predicted_class":r.predicted_class,"confidence":float(r.confidence),"entropy":float(r.entropy),"margin":float(r.margin),"trust_score":float(r.trust_score),"trust_level":r.trust_level,"human_review_required":bool(r.requires_human_review),"human_decision":f["human_decision"] if f else "NOT REQUIRED","corrected_class":f["corrected_class"] if f else None,"feedback_persisted":bool(f) and FEEDBACK_JSONL.exists(),"saved_llm_explanation_available":not exp.startswith("No saved"),"gradcam_generated_dynamically":True})
summary=pd.DataFrame(rows); display(summary)
summary_path=HF_DIR/"four_sample_hitl_demo_summary.csv"; summary.to_csv(summary_path,index=False); print("Summary saved:",summary_path)


## Notes / troubleshooting

- Notebook 4 loads `processed/test.pt` as tensors and creates the report with `image_index = np.arange(len(test_y))`, using a DataLoader with `shuffle=False`. This notebook now uses that exact same tensor row order, so no image manifest or manual path mapping is required.
- The displayed image is inverse-normalized from the processed tensor using the ImageNet mean/std recorded in Notebook 4. The model receives the original normalized tensor.
- Grad-CAM is generated dynamically from the loaded ResNet50 `layer4[-1]`; no saved Grad-CAM image manifest is required.
- Saved Notebook 6 explanations are displayed when available. If no matching saved explanation exists, the notebook states that explicitly instead of fabricating a Qwen response.
- The three low-trust cases use the configured trust thresholds and the high-trust case is selected from samples that do not trigger a review condition.
- Feedback is appended to JSONL and merged into CSV without deleting prior history. Avoid rerunning the interactive feedback cell unless you intend to collect additional reviews.
